# Phase 3: LoRA-Mistral-7B Training (QLoRA, 4-bit)

**CSCI E-222 · Spring 2026**

Trains a parameter-efficient LoRA adapter on Mistral-7B-Instruct-v0.2 for multi-label product tagging.
The backbone is frozen in 4-bit; only LoRA weights (`q_proj`, `v_proj`) and the classification head train.

**Expected runtime on A100:** 3–6 hrs · **Expected peak VRAM:** ~18–24 GB

Results are compared directly against the BERT baseline from Phase 2.

In [ ]:
import sys, os
from pathlib import Path

_here = Path(os.getcwd())
if _here.name != 'notebooks':
    _nb = _here / 'notebooks'
    if _nb.exists():
        os.chdir(_nb)

sys.path.insert(0, str(Path('..').resolve()))

import json
import logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from torch.utils.data import DataLoader

from data.product_dataset import ProductDataset
from models.lora_mistral import build_model_and_tokenizer, LoraMistralTrainer, LoraMistralClassifier, measure_latency
from eval.metrics import compute_metrics, per_label_f1, compare_models

logging.basicConfig(level=logging.INFO, format='%(levelname)s | %(message)s')
sns.set_theme(style='whitegrid', palette='muted')

print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 1. Load Taxonomy & Stats

In [ ]:
with open('../data/taxonomy.json') as f:
    taxonomy = json.load(f)
LABEL_NAMES = [l['tag'] for l in taxonomy['labels']]
NUM_LABELS  = len(LABEL_NAMES)

with open('../data/processed/stats.json') as f:
    stats = json.load(f)
POS_WEIGHTS = stats['per_label_pos_weights']

print(f'Labels: {NUM_LABELS} | Train: {stats["n_train"]:,} | Val: {stats["n_val"]:,}')

## 2. Training Configuration

Batch size is kept small (4) due to Mistral's memory footprint.
Gradient accumulation over 8 steps gives an effective batch of 32 — matching the BERT baseline.

In [ ]:
BATCH_SIZE     = 4       # physical batch — Mistral is memory-heavy
GRAD_ACCUM     = 8       # effective batch = 32 (matches BERT baseline)
NUM_EPOCHS     = 3       # QLoRA converges faster than full fine-tuning
MAX_LENGTH     = 120
CHECKPOINT_DIR = '../checkpoints/lora_mistral'

TRAIN_CONFIG = {
    'learning_rate':       2e-4,   # higher than BERT — LoRA params only
    'weight_decay':        0.01,
    'num_epochs':          NUM_EPOCHS,
    'max_grad_norm':       1.0,
    'batch_size':          BATCH_SIZE,
    'grad_accum_steps':    GRAD_ACCUM,
    'max_length':          MAX_LENGTH,
    'lora_r':              16,
    'lora_alpha':          32,
    'lora_dropout':        0.1,
    'target_modules':      ['q_proj', 'v_proj'],
}

print(json.dumps(TRAIN_CONFIG, indent=2))

## 3. Build Model & Tokenizer

In [ ]:
model, tokenizer = build_model_and_tokenizer(
    num_labels=NUM_LABELS,
    lora_r=TRAIN_CONFIG['lora_r'],
    lora_alpha=TRAIN_CONFIG['lora_alpha'],
    lora_dropout=TRAIN_CONFIG['lora_dropout'],
)

if torch.cuda.is_available():
    mem = torch.cuda.memory_allocated() / 1e9
    print(f'GPU memory after model load: {mem:.2f} GB')

## 4. DataLoaders

Reuses `ProductDataset` from Phase 2 — same split files, same tokenization interface.
The Mistral tokenizer replaces the BERT tokenizer transparently.

In [ ]:
train_ds = ProductDataset('../data/processed/train.parquet', tokenizer, MAX_LENGTH, NUM_LABELS)
val_ds   = ProductDataset('../data/processed/val.parquet',   tokenizer, MAX_LENGTH, NUM_LABELS)
test_ds  = ProductDataset('../data/processed/test.parquet',  tokenizer, MAX_LENGTH, NUM_LABELS)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')
print(f'Effective batch size: {BATCH_SIZE * GRAD_ACCUM} (after gradient accumulation)')

## 5. Train

In [ ]:
trainer = LoraMistralTrainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    config=TRAIN_CONFIG,
    checkpoint_dir=CHECKPOINT_DIR,
    pos_weight=POS_WEIGHTS,
)

results = trainer.train()
print(f"\nBest Micro-F1: {results['best_micro_f1']:.4f}")

## 6. Loss & F1 Curves

In [ ]:
history = results['history']
epochs  = range(1, NUM_EPOCHS + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs, history['train_loss'], label='Train loss', marker='o')
axes[0].plot(epochs, history['val_loss'],   label='Val loss',   marker='s')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('BCEWithLogitsLoss')
axes[0].set_title('LoRA-Mistral-7B — Loss')
axes[0].legend()

axes[1].plot(epochs, history['micro_f1'], label='Micro-F1', marker='o')
axes[1].plot(epochs, history['macro_f1'], label='Macro-F1', marker='s')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('F1')
axes[1].set_title('LoRA-Mistral-7B — Validation F1')
axes[1].legend()

plt.tight_layout()
plt.savefig('../data/processed/lora_mistral_curves.png', dpi=150)
plt.show()

## 7. Evaluate Best Checkpoint on Test Set

In [ ]:
best_model = LoraMistralClassifier.from_checkpoint(CHECKPOINT_DIR, num_labels=NUM_LABELS)
best_model.eval()
device = 'cuda' if torch.cuda.is_available() else 'cpu'

best_thresholds = results['best_thresholds']

all_probs, all_labels = [], []
with torch.no_grad():
    for batch in test_loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        logits = best_model(input_ids, attention_mask)
        probs  = torch.sigmoid(logits).cpu().float().numpy()
        all_probs.append(probs)
        all_labels.append(batch['labels'].numpy())

y_prob = np.vstack(all_probs)
y_true = np.vstack(all_labels)

lora_metrics = compute_metrics(y_true, y_prob, best_thresholds)

print('LoRA-Mistral test set results:')
for k, v in lora_metrics.items():
    if k != 'per_label_f1':
        print(f'  {k:<20} {v}')

## 8. Head-to-Head Comparison with BERT Baseline

In [ ]:
# Load BERT test results saved in Phase 2
# (Re-run Phase 2 eval cell if metrics weren't persisted)
bert_model_name = 'bert-base-uncased'

# Summary table
summary_rows = []
metrics_to_show = ['micro_f1', 'macro_f1', 'hamming_loss', 'coverage_error', 'ranking_loss']

# You'll have bert_metrics from Phase 2 in the same Colab session,
# or reload the checkpoint and re-evaluate here.
# For now, this cell expects `bert_metrics` to be defined.
try:
    comparison = compare_models(
        {'BERT': bert_metrics, 'LoRA-Mistral': lora_metrics},
        LABEL_NAMES
    )
    summary_df = pd.DataFrame(comparison['summary']).T
    print('Model comparison (test set):')
    print(summary_df.to_string())
    summary_df.to_csv('../data/processed/model_comparison.csv')
except NameError:
    print('bert_metrics not in scope — run Phase 2 eval in the same session or reload the checkpoint.')

## 9. Per-Label F1 Comparison Heatmap

In [ ]:
try:
    heatmap_data = pd.DataFrame({
        'BERT':         bert_metrics['per_label_f1'],
        'LoRA-Mistral': lora_metrics['per_label_f1'],
    }, index=LABEL_NAMES)

    # Sort by average F1 descending
    heatmap_data['avg'] = heatmap_data.mean(axis=1)
    heatmap_data = heatmap_data.sort_values('avg', ascending=False).drop(columns='avg')

    fig, ax = plt.subplots(figsize=(7, 11))
    sns.heatmap(
        heatmap_data,
        annot=True, fmt='.2f',
        cmap='RdYlGn', vmin=0, vmax=1,
        linewidths=0.5, ax=ax,
    )
    ax.set_title('Per-label F1: BERT vs LoRA-Mistral (test set)')
    ax.set_xlabel('Model')
    plt.tight_layout()
    plt.savefig('../data/processed/per_label_f1_heatmap_phase3.png', dpi=150)
    plt.show()
except NameError:
    print('bert_metrics not in scope — skipping heatmap.')

## 10. Latency Comparison

This gap directly informs the Model Selector's confidence threshold in Phase 5:
the higher the latency penalty for routing to Mistral, the tighter the threshold needs to be.

In [ ]:
lora_latency = measure_latency(best_model, tokenizer, n_samples=50, max_length=MAX_LENGTH)

print('LoRA-Mistral inference latency (single sample):')
for k, v in lora_latency.items():
    print(f'  {k:<12} {v} ms')

with open('../data/processed/lora_mistral_latency.json', 'w') as f:
    json.dump({'model': 'lora_mistral', **lora_latency}, f, indent=2)

# Load BERT latency for comparison
try:
    with open(f'../data/processed/{bert_model_name}_latency.json') as f:
        bert_latency = json.load(f)

    print(f'\nLatency ratio (Mistral / BERT):')
    for k in ['mean_ms', 'p95_ms']:
        ratio = lora_latency[k] / bert_latency[k]
        print(f'  {k:<12} {ratio:.1f}x slower')
except FileNotFoundError:
    print('BERT latency file not found — run Phase 2 notebook first.')

In [ ]:
print('Phase 3 complete.')
print(f'LoRA adapter + classifier head saved to: {CHECKPOINT_DIR}')
print('Latency gap logged — use it to calibrate the Model Selector threshold in Phase 5.')